# Estruturação da Bíblia em versículos para o vectorstore

Este notebook prepara a Bíblia Ave-Maria para uso em busca semântica e recuperação de contexto. Ele extrai o texto do PDF, remove ruídos de formatação, organiza o conteúdo em livro, capítulo e versículo, e gera uma base mais confiável para indexação.

## Objetivos

- Extrair o texto bruto do PDF ou reutilizar o TXT já processado.
- Remover cabeçalhos, rodapés e trechos indesejados.
- Identificar livros, capítulos e versículos com consistência.
- Produzir uma saída estruturada para o pipeline de vectorstore.


In [1]:
BIBLIA_INPUT_PDF = (
    "../../data/raw/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.pdf"
)
BIBLIA_TXT_EXTRACTED = (
    "../../data/raw/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.txt"
)
BIBLIA_TEMP = (
    "../../data/processed/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible-temp.txt"
)
BIBLIA_OUTPUT = (
    "../../data/processed/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible-verses.csv"
)

In [2]:
PROCESSAR_PDF = False
PROCESSAR_TXT = True

In [3]:
import pdfplumber


def extrair_texto_pdf(pdf_path, txt_path):
    with pdfplumber.open(pdf_path) as pdf:
        with open(txt_path, "w", encoding="utf-8") as f_out:
            for i, page in enumerate(pdf.pages):
                if i < 2:
                    continue
                elif i == 2904:
                    break
                texto = page.extract_text()
                if texto:
                    f_out.write(texto + "\n")
    print(f"Texto extraído para: {txt_path}")


if PROCESSAR_PDF:
    extrair_texto_pdf(BIBLIA_INPUT_PDF, BIBLIA_TXT_EXTRACTED)

In [4]:
def remover_3_primeiras_linhas(caminho_entrada, caminho_saida):
    with open(caminho_entrada, "r", encoding="utf-8") as f_in:
        linhas = f_in.readlines()
    with open(caminho_saida, "w", encoding="utf-8") as f_out:
        f_out.writelines(linhas[3:])
    print(f"Texto processado para: {caminho_entrada}")


if PROCESSAR_TXT:
    remover_3_primeiras_linhas(BIBLIA_TXT_EXTRACTED, BIBLIA_TEMP)

Texto processado para: ../../data/raw/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.txt


In [5]:
# def remover_trechos_indesejados(caminho):
#     trecho_alvo_1 = "Bíblia Ave-Maria"
#     trecho_alvo_2 = "© 1959 Editora Ave-Maria."
#     remover_linhas = False

#     with open(caminho, "r", encoding="utf-8") as f:
#         linhas = f.readlines()

#     linhas_filtradas = []
#     for linha in linhas:
#         if trecho_alvo_1 in linha:
#             remover_linhas = True
#         elif remover_linhas and trecho_alvo_2 in linha:
#             continue
#         elif remover_linhas and linha.strip().isdigit():
#             remover_linhas = False
#             continue
#         if not remover_linhas:
#             linhas_filtradas.append(linha)

#     with open(caminho, "w", encoding="utf-8") as f:
#         f.writelines(linhas_filtradas)

#     print(f"Trechos indesejados removidos de {caminho}")


# if PROCESSAR_TXT:
#     remover_trechos_indesejados(BIBLIA_TEMP)

In [6]:
map_proximo_livro = {
    "Livro Inicial": "Gênesis",
    # Antigo Testamento
    "Gênesis": "Êxodo",
    "Êxodo": "Levítico",
    "Levítico": "Números",
    "Números": "Deuteronômio",
    "Deuteronômio": "Josué",
    "Josué": "Juízes",
    "Juízes": "Rute",
    "Rute": "1 Samuel",
    "1 Samuel": "2 Samuel",
    "2 Samuel": "1 Reis",
    "1 Reis": "2 Reis",
    "2 Reis": "1 Crônicas",
    "1 Crônicas": "2 Crônicas",
    "2 Crônicas": "Esdras",
    "Esdras": "Neemias",
    "Neemias": "Tobias",
    "Tobias": "Judite",
    "Judite": "Ester",
    "Ester": "Jó",
    "Jó": "Salmos",
    "Salmos": "1 Macabeus",
    "1 Macabeus": "2 Macabeus",
    "2 Macabeus": "Provérbios",
    "Provérbios": "Eclesiastes",
    "Eclesiastes": "Cântico dos Cânticos",
    "Cântico dos Cânticos": "Sabedoria",
    "Sabedoria": "Eclesiástico",
    "Eclesiástico": "Isaías",
    "Isaías": "Jeremias",
    "Jeremias": "Lamentações de Jeremias",
    "Lamentações de Jeremias": "Baruc",
    "Baruc": "Ezequiel",
    "Ezequiel": "Daniel",
    "Daniel": "Oseias",
    "Oseias": "Joel",
    "Joel": "Amós",
    "Amós": "Abdias",
    "Abdias": "Jonas",
    "Jonas": "Miqueias",
    "Miqueias": "Naum",
    "Naum": "Habacuc",
    "Habacuc": "Sofonias",
    "Sofonias": "Ageu",
    "Ageu": "Zacarias",
    "Zacarias": "Malaquias",
    "Malaquias": "São Mateus",
    # Novo Testamento
    "São Mateus": "São Marcos",
    "São Marcos": "São Lucas",
    "São Lucas": "São João",
    "São João": "Atos",
    "Atos": "Romanos",
    "Romanos": "1 Coríntios",
    "1 Coríntios": "2 Coríntios",
    "2 Coríntios": "Gálatas",
    "Gálatas": "Efésios",
    "Efésios": "Filipenses",
    "Filipenses": "Colossenses",
    "Colossenses": "1 Tessalonicenses",
    "1 Tessalonicenses": "2 Tessalonicenses",
    "2 Tessalonicenses": "1 Timóteo",
    "1 Timóteo": "2 Timóteo",
    "2 Timóteo": "Tito",
    "Tito": "Filemon",
    "Filemon": "Hebreus",
    "Hebreus": "Tiago",
    "Tiago": "1 São Pedro",
    "1 São Pedro": "2 São Pedro",
    "2 São Pedro": "1 São João",
    "1 São João": "2 São João",
    "2 São João": "3 São João",
    "3 São João": "São Judas",
    "São Judas": "Apocalipse",
}

In [7]:
import pandas as pd


def adicionar_versiculo(
    df: pd.DataFrame,
    book: str,
    chapter: int,
    verse_start: int | None,
    verse_end: int | None,
    text: str,
    verse_acc: int | None,
    pdf_page: int,
    need_review: bool,
    raw_verse_marker: str,
    parse_issue: str | None,
) -> pd.DataFrame:
    """Adiciona versículo ao dataframe"""    
    new_row = pd.DataFrame(
        [
            {
                "book": book,
                "chapter": chapter,
                "verse_start": verse_start,
                "verse_end": verse_end,
                "text": text,
                "verse_acc": verse_acc,
                "pdf_page": pdf_page,
                "need_review": need_review,
                "raw_verse_marker": raw_verse_marker,
                "parse_issue": parse_issue,
            }
        ]
    )
    
    # print("Adicionando versículo: {book} {chapter}:{verse_start}-{verse_end} - {text}".format(
    #     book=book, chapter=chapter, verse_start=verse_start, verse_end=verse_end, text=text
    # ))
    
    return pd.concat([df, new_row], ignore_index=True)


def escrever_versiculo_no_arquivo():
    """Escreve versículo em arquivo.

    Formato do versículo: nome_livro numero_capitulo:nume_versiculo texto_versiculo
    """
    pass

# Estruturação confiável de versículos bíblicos

Para elaborar um parser melhor, eu seguiria estas instruções:

- [ ] Defina o modelo de saída antes de extrair: cada registro deve ter livro canônico, capítulo, versículo, texto, página do PDF, linha bruta e um indicador de confiança.
- [ ] Extraia preservando contexto de página. Cada fragmento deve carregar o número da página e, idealmente, posição/ordem no PDF, para que qualquer erro seja rastreável até a imagem original.
- [ ] Normalize a fonte com cautela: remova cabeçalho, rodapé e número de página apenas quando coincidirem com padrões confirmados e posição de página; não descarte toda linha numérica genericamente.
- [ ] Reconheça explicitamente os tipos de marcador: capítulo (`nome de livro + capítulo`), versículo simples, intervalo (`2-7`), faixa compacta ou agrupada, títulos, notas e referências cruzadas. Qualquer linha que não se enquadre deve virar texto ou ir para revisão, nunca marcador por suposição.
- [ ] Mantenha intervalos como intervalos. Para `Josué 19:2-7`, prefira guardar um único registro com início 2 e fim 7, ou dividir em `vv. 2–7` somente se o PDF trouxer separação textual confiável. Não invente a distribuição.
- [ ] Valide a sequência em cada capítulo. Registre alertas para repetição, salto, retrocesso, intervalo sobreposto e número improvável; a validação deve apontar `8 → 10 → 10`, não corrigir automaticamente.
- [ ] Tenha regras explícitas para exceções bíblicas: Salmos, livros de um capítulo, títulos de salmos, versículos agrupados, notas e variações de nomenclatura dos livros.
- [ ] Preserve o original e a versão limpa. Nunca substitua o texto bruto: mantenha uma camada de extração, uma de segmentação e outra de normalização, para que uma correção não exija reextrair tudo.
- [ ] Crie uma fila de revisão humana para casos ambíguos. Em vez de decidir silenciosamente, marque registros com baixa confiança, como `2-7` convertido em `27` ou repetições de número.
- [ ] Compare a estrutura final com uma referência canônica de contagem por livro/capítulo. A meta não é substituir o texto da Ave-Maria, mas verificar se as chaves de referência estão completas e coerentes.
- [ ] Faça testes de regressão com páginas problemáticas: `Josué 19`, `2 Reis 3`, páginas com rodapé, quebras por hifenização, início/fim de livro e páginas com títulos. Cada erro corrigido deve virar um caso permanente de teste.
- [ ] Mantenha a base canônica em registros atômicos por versículo, com ID estável (`tradução`, livro, capítulo, versículo), e represente `2-7` como uma referência/ocorrência que aponta para os IDs `2`, `3`, `4`, `5`, `6` e `7`.
- [ ] Se o texto fonte não permitir separar com segurança cada versículo, marque o trecho como `NEEDS_REVIEW`, preservando o intervalo e a fonte bruta, em vez de inventar a divisão.
- [ ] Use uma fonte católica, licenciada e versionada, não apenas "canônica".
- [ ] Valide comparando SQLite e Chroma, garantindo que o índice seja recriado somente a partir da base certificada.

## Critérios de conclusão

- [ ] A extração preserva o original, a versão limpa, a rastreabilidade por página e a fila de revisão humana.
- [ ] A base canônica permanece atômica por versículo, com ocorrências de intervalos mapeadas sem inventar divisão textual.
- [ ] SQLite e Chroma batem exatamente contra a base certificada antes de qualquer publicação ou reindexação.


In [8]:
import re
import os

from bible_model import Verse


def transformar_biblia_versiculo_a_versiculo(caminho_entrada, caminho_saida):
    livro_atual = "Livro Inicial"
    capitulo_atual = 0
    verse_start_atual = None
    verse_end_atual = None
    ultimo_verse_end = None
    raw_verse_marker_atual = None
    parse_issue_atual = None
    verse_acc_atual = None
    versiculo_acc = 0
    mark_new_page = False
    pagina_atual = 3

    buffer_versiculo = []

    df_verse = pd.DataFrame(
        {
            "book": pd.Series(dtype="Int64"),
            "chapter": pd.Series(dtype="Int64"),
            "verse_start": pd.Series(dtype="Int64"),
            "verse_end": pd.Series(dtype="Int64"),
            "text": pd.Series(dtype="string"),
            "verse_acc": pd.Series(dtype="Int64"),
            "pdf_page": pd.Series(dtype="Int64"),
            "need_review": pd.Series(dtype="boolean"),
            "raw_verse_marker": pd.Series(dtype="string"),
            "parse_issue": pd.Series(dtype="string"),
        }
    )

    # Regex para detectar linhas como "Gênesis 1"
    # Esse regex é capaz de detectar trechos como:
    # - "Gênesis 1"
    # - "1 Samuel 2"
    # - "Cântico dos Cânticos 3"
    regex_livro_capitulo = re.compile(r"^(.*)\s+(\d+)$")
    regex_intervalo_versiculo = re.compile(r"^(\d+)\s*-\s*(\d+)$")
    regex_intervalo_embutido = re.compile(
        r"^(.*?:)\s+(\d+)\s*-\s*(\d+)\s+(.+)$"
    )

    def salvar_buffer_versiculo():
        nonlocal df_verse, buffer_versiculo

        if raw_verse_marker_atual is None or not buffer_versiculo:
            buffer_versiculo = []
            return

        # Alguns versículos ficam quebrados em várias linhas no texto bruto.
        # O buffer só é persistido quando o próximo marcador aparece.
        df_verse = adicionar_versiculo(
            df_verse,
            livro_atual,
            capitulo_atual,
            verse_start_atual,
            verse_end_atual,
            "\n".join(buffer_versiculo),
            verse_acc_atual,
            pagina_atual,
            parse_issue_atual is not None,
            raw_verse_marker_atual,
            parse_issue_atual,
        )
        buffer_versiculo = []

    def iniciar_novo_versiculo(
        raw_marker, explicit_start=None, explicit_end=None
    ):
        nonlocal verse_start_atual, verse_end_atual, ultimo_verse_end
        nonlocal raw_verse_marker_atual, parse_issue_atual, verse_acc_atual
        nonlocal versiculo_acc, buffer_versiculo

        proximo_esperado = (
            1 if ultimo_verse_end is None else ultimo_verse_end + 1
        )

        # O primeiro marcador pode ter sido adiantado por um erro na fonte.
        # Só recuperamos o estado quando o mesmo número aparece novamente:
        # os dois blocos são mantidos juntos, sem criar o versículo ausente.
        if (
            explicit_start is None
            and raw_verse_marker_atual == raw_marker
            and parse_issue_atual == "unexpected_marker"
            and ultimo_verse_end is not None
            and int(raw_marker) == proximo_esperado + 1
        ):
            verse_start_atual = int(raw_marker)
            verse_end_atual = int(raw_marker)
            raw_verse_marker_atual = f"{raw_marker} | {raw_marker}"
            parse_issue_atual = "merged_duplicate_marker"
            verse_acc_atual = versiculo_acc + 1
            versiculo_acc += 1
            ultimo_verse_end = verse_end_atual
            return

        # Um salto de um número seguido pela sequência indica que a fonte
        # omitiu apenas o marcador intermediário. A duplicação acima tem
        # prioridade para não inferir um versículo ausente quando o mesmo
        # marcador foi repetido.
        if (
            explicit_start is None
            and raw_verse_marker_atual is not None
            and raw_verse_marker_atual.isdigit()
            and parse_issue_atual == "unexpected_marker"
            and ultimo_verse_end is not None
            and int(raw_verse_marker_atual) == proximo_esperado + 1
            and int(raw_marker) == int(raw_verse_marker_atual) + 1
        ):
            verse_start_atual = int(raw_verse_marker_atual)
            verse_end_atual = int(raw_verse_marker_atual)
            parse_issue_atual = "skipped_verse_marker"
            verse_acc_atual = versiculo_acc + 1
            versiculo_acc += 1
            ultimo_verse_end = verse_end_atual
            proximo_esperado = ultimo_verse_end + 1

        salvar_buffer_versiculo()
        raw_verse_marker_atual = raw_marker
        verse_start_atual = None
        verse_end_atual = None
        parse_issue_atual = None
        verse_acc_atual = None

        if explicit_start is not None:
            if (
                explicit_start == proximo_esperado
                and explicit_end >= explicit_start
            ):
                verse_start_atual = explicit_start
                verse_end_atual = explicit_end
                parse_issue_atual = "explicit_verse_range"
        else:
            numero_versiculo = int(raw_marker)
            if numero_versiculo == proximo_esperado:
                verse_start_atual = numero_versiculo
                verse_end_atual = numero_versiculo
            elif raw_marker.startswith(
                str(proximo_esperado)
            ):
                possible_end = raw_marker[len(str(proximo_esperado)) :]
                if possible_end and int(possible_end) >= proximo_esperado:
                    verse_start_atual = proximo_esperado
                    verse_end_atual = int(possible_end)
                    parse_issue_atual = "merged_verse_marker"

            # Em um salto maior, o marcador explícito da fonte continua
            # sendo válido. Aceitamo-lo como versículo simples e deixamos
            # os números ausentes sem registro, sem inventar intervalos.
            if (
                verse_start_atual is None
                and numero_versiculo > proximo_esperado + 1
            ):
                verse_start_atual = numero_versiculo
                verse_end_atual = numero_versiculo
                parse_issue_atual = "skipped_verse_marker"

        if verse_start_atual is None:
            parse_issue_atual = "unexpected_marker"
        else:
            verse_acc_atual = versiculo_acc + 1
            versiculo_acc += verse_end_atual - verse_start_atual + 1
            ultimo_verse_end = verse_end_atual

        buffer_versiculo = []

    with open(caminho_entrada, "r", encoding="utf-8") as entrada:
        for linha in entrada:
            linha = linha.strip()

            if not linha:
                continue  # pula linhas vazias

            # print(f"Linha atual: {linha}")

            # Verifica nova página (PDF)
            if linha == "Bíblia Ave-Maria" or linha == "© 1959 Editora Ave-Maria.":
                # A troca de página pode acontecer no meio de um versículo.
                # Nesse caso, mantemos o buffer aberto para unificar o texto.
                mark_new_page = True
                continue
            if mark_new_page and linha.isdigit():
                pagina_atual = int(linha)
                # print(f"Nova página detectada: {pagina_atual}")
                mark_new_page = False
                continue

            # Verifica se é início de livro (só nome do livro)
            if (
                livro_atual
                != "Apocalipse"  # Apocalipse é o último livro, não há próximo mapeado!
                and linha.startswith(map_proximo_livro[livro_atual])
                and len(linha.split()) <= 3
            ):
                salvar_buffer_versiculo()
                print(f"Livro encontrado: {linha}")
                livro_atual = linha.strip()
                capitulo_atual = 0
                verse_start_atual = None
                verse_end_atual = None
                ultimo_verse_end = None
                raw_verse_marker_atual = None
                parse_issue_atual = None
                verse_acc_atual = None
                buffer_versiculo = []
                continue

            # Verifica se é início de capítulo ("Livro Capítulo")
            match = regex_livro_capitulo.match(linha)
            if match:
                # print(f"Capítulo encontrado: {match.group(1)} {match.group(2)}")
                salvar_buffer_versiculo()
                capitulo_atual = int(match.group(2).strip())
                verse_start_atual = None
                verse_end_atual = None
                ultimo_verse_end = None
                raw_verse_marker_atual = None
                parse_issue_atual = None
                verse_acc_atual = None
                buffer_versiculo = []
                continue

            # Verifica se é um intervalo explícito de versículos, como "35-39".
            match_intervalo = regex_intervalo_versiculo.match(linha)
            if match_intervalo:
                iniciar_novo_versiculo(
                    linha,
                    int(match_intervalo.group(1)),
                    int(match_intervalo.group(2)),
                )
                continue

            # Verifica se é um número (versículo)
            elif linha.isdigit():
                # print(f"Número encontrado: {linha}"
                iniciar_novo_versiculo(linha.strip())
                continue
            
            # Daqui pra frente estamos lidando com 1 ou mais linhas que formam um único versículo.
            # As quebras de linha internas são preservadas até surgir o próximo marcador.
            if raw_verse_marker_atual is None:
                continue

            # Algumas faixas foram extraídas na mesma linha do versículo
            # anterior, como "... filhos: 19-32 Dentre ...". Só as
            # separamos quando a faixa continua exatamente a sequência atual.
            match_intervalo_embutido = regex_intervalo_embutido.match(linha)
            if (
                match_intervalo_embutido
                and verse_end_atual is not None
                and int(match_intervalo_embutido.group(2))
                == verse_end_atual + 1
                and int(match_intervalo_embutido.group(3))
                >= int(match_intervalo_embutido.group(2))
            ):
                buffer_versiculo.append(match_intervalo_embutido.group(1))
                iniciar_novo_versiculo(
                    f"{match_intervalo_embutido.group(2)}-"
                    f"{match_intervalo_embutido.group(3)}",
                    int(match_intervalo_embutido.group(2)),
                    int(match_intervalo_embutido.group(3)),
                )
                buffer_versiculo.append(match_intervalo_embutido.group(4))
                continue

            buffer_versiculo.append(linha)

    # Salva o último versículo
    salvar_buffer_versiculo()

    df_verse.to_csv(caminho_saida, index=False)
    print(f"Transformação concluída. Versículos salvos em {caminho_saida}")
    return df_verse


In [ ]:
if PROCESSAR_TXT:
    df_verse = transformar_biblia_versiculo_a_versiculo(
        BIBLIA_TEMP,
        BIBLIA_OUTPUT,
    )

    if PROCESSAR_TXT and os.path.exists(BIBLIA_TEMP):
        os.remove(BIBLIA_TEMP)
        print(f"Arquivo temporário removido: {BIBLIA_TEMP}")

Livro encontrado: Gênesis
Livro encontrado: Êxodo


## Análise e Pós-processamento

In [ ]:
if not PROCESSAR_TXT:
    raise ValueError(
        "PROCESSAR_TXT Falso, PARE O NOTEBOOK AQUI!\\Não prosseguir com análise e pós-processamento."
    )

In [ ]:
df_verse

In [ ]:
df_verse.head(10)

In [ ]:
df_verse.tail(10)

In [ ]:
df_verse.info()

In [ ]:
review_columns = [
    "book",
    "chapter",
    "verse_start",
    "verse_end",
    "raw_verse_marker",
    "parse_issue",
    "verse_acc",
    "pdf_page",
    "text",
]
df_verse.loc[df_verse["need_review"].fillna(False), review_columns]

In [ ]:
# Traz todos os need_review, junto à linha anterior e à próxima linha, já deduplicando caso choque
need_review_pos = df_verse.index[df_verse["need_review"].fillna(False)]

context_pos = set()
for pos in need_review_pos:
    context_pos.update({pos - 1, pos, pos + 1})

context_pos = [pos for pos in sorted(context_pos) if 0 <= pos < len(df_verse)]
df_verse.iloc[context_pos][review_columns]

In [ ]:
df_verse["parse_issue"].value_counts()

In [ ]:
# Traz todos os raw_verse_marker que não foram parseados corretamente, junto à linha anterior e à próxima linha, já deduplicando caso choque
parse_issue_pos = df_verse.index[df_verse["parse_issue"].notna()]

context_pos = set()
for pos in parse_issue_pos:
    context_pos.update({pos - 1, pos, pos + 1})

context_pos = [pos for pos in sorted(context_pos) if 0 <= pos < len(df_verse)]
df_verse.iloc[context_pos][review_columns]

In [ ]:
# Traz todos os issues to tipo skipped_verse_marker, junto à linha anterior e à próxima linha, já deduplicando caso choque
parse_issue_pos = df_verse.index[df_verse["parse_issue"].eq("skipped_verse_marker")]

context_pos = set()
for pos in parse_issue_pos:
    context_pos.update({pos - 1, pos, pos + 1})

context_pos = [pos for pos in sorted(context_pos) if 0 <= pos < len(df_verse)]
df_verse.iloc[context_pos][review_columns]